## SSL / TLS

- One central problem of communicating between different computers is trust; how can we know that our machine is speaking to the machine it is claiming to be? And how does the other machine know that we are who we claim to be?

- SSL/TLS are 2 approaches developed to handle this trust problem. This is best illustrated with examples, so we will walk through 2 use cases:
    1. Websurfing
    2. Flink & Kafka


### SSL/TLS in Websurfing

#### Problem set up

- In surfing the web, on the user side, people only interact with plaintext URLs
    - For example, you may type "reddit.com" in your URL bar

- Your browser goes to the DNS server to look up the IP address of your desired webpage

- Now armed with the IP address. You can now hit reddit's web server. OR CAN YOU? There are still 3 concerns:
    - How do you know you're truly hitting reddit, and not some malicious server that has tricked the DNS into redirecting you there?
    - Even if you are talking to reddit, you may intend to exchange some sensitive information between your machine and reddit's server. How do you know that there isn't someone in the middle of that connection?
    - In the same vein, how do you know that the data you receive from reddit **truly** came from reddit, and was not modified halfway by some malicious actor?

#### Solution

- The problems identified above boil down to 2 issues; **trust** and **encryption**. SSL/TLS address both of these issues. How does SSL/TLS do this?

- Process
    1. **Client Hello:** Your browser sends a message to the server, containing: 
        - The TLS versions it supports
        - The list of encryption algorithms it knows (also know as the **Cipher Suite**)
        - A random string, known as the **client random** string
    
    2. **Server Hello:** Now, the server does several things with this information
        - It chooses the strongest TLS version and Cipher Suite available that are mutually compatible
        - The server sends its public certificate, which is signed by a certificate authority (e,g, Digicert, Let's Encrypt). The list of "pre-approved" authorities are maintained by companies like Google, Apple, etc
        - A random string, known as the **server random** string
    
    3. **Background Check:** Upon receiving the `server hello`, your browser now takes the public certificate, and checks a few things:
        - Check signature against the built in trust store, pre-approved by Google/Apple etc on whatever device you are using
        - Checks that expiration date and domain name on the certificate matches the website you're visiting
        - If this fails, this is when you see the "Your connection is not private" warning in browser
    
    4. **Trust Established, Exchange Data:** If the previous step passes, we now know that we are communicating with someone legit. This fulfills the **trust** objective of the SSL/TSL handshake. 
        - However, we still need to ensure the **encryption** objective!
        - The browser generates a third random string, called the **pre-master secret**
        - Using the server's public key (from the certificate), the client (browser) encrypts this string, and sends it to the server
        - The server tries to decrypt the random string with the private key matching the public key on the certificate
        - Since **ONLY** the server contains the matching private key, only the server can decrypt this string
    
    5. **Generate session keys:** Now both the client and servers have 3 sets of strings; the client random, server random, and pre-master secret. These are used to generate a **master secret**, which is then used to generate **session keys**
    
    6. **Private mode:**
        - Using the session keys generated that are theoretically only available to both the client AND the server, you can now send encrypted data using these keys, and these can be decrypted only within the session connection
        - Both sides will send a "FINISHED" message that is encrypted with the new keys, and will try to decrypt them
        - If both sides successfully decrypt the message, a secure session is created successfully

#### Notes

- Notice that the process above only requires that servers prove themselves to clients. Servers don't care about client identity! This is therefore also known as **one-way TLS**

- How does this prevent man-in-the-middle issues?
    - Let's suppose I set up something to maliciously pretend I am reddit
        - I somehow poison the global DNS such that all traffic to reddit.com routes to me first, and I forward the message to the actual reddit server. Similarly, all messages from reddit come to me, and I pass it on to the client
        - Without TLS, I am able to read everything that client sends to reddit, and reddit sends to client!
    - What happens with TLS?
        - At step 2, my malicious server needs to provide the browser with a certificate that contains a public key. 
            - I could forward reddit's real certificate. But then I will fail in step 4 when I try to decrypt the pre-master secret, because I will not have the matching private key in my server. That means that all encrypted messages between reddit and client will become gibberish, EVEN THOUGH I AM IN THE MIDDLE
            - I could forward a fake certificate with my own keys. But then this certificate will fail the background check, and client browser will surface a warning

### SSL/TLS in Kafka/Flink

#### Problem setup

- We have already seen how a one-way TLS works. That suffices for situations like web browsing, because while clients care what server is they are connecting to, the server doesn't care who is connecting to it. It serves traffic either way, because information is not private!

- This assumption fails in Kafka/Flink set up:
    - Let's assume we have a Kafka cluster with some distributed brokers
    - Let's assume we have written a flink app with distributed Jobmanagers and Taskmanagers
    - The flink app needs to retrieve data from the brokers, and needs to pass the processed data back to the brokers to publish on some topic
    - There are now 2 points of failures! 
        1. How do we know the flink app requesting data is not a malicious actor trying to steal data?
        2. How do we know the broker the flink app pushes to is not a malicious actor trying to steal data?
    

#### Solution

- We do the same thing now, but bidirectionally!

- Process
    1. **Client Hello:** Your Flink app sends a message to the Kafka broker, containing: 
        - The TLS versions it supports
        - The list of encryption algorithms it knows (also know as the **Cipher Suite**)
        - A random string, known as the **client random** string
    
    2. **Server Hello:** Once again, the broker does several things with this information
        - It chooses the strongest TLS version and Cipher Suite available that are mutually compatible
        - The server sends its public certificate with a "public key", which is signed by a certificate authority (e,g, Digicert, Let's Encrypt). The list of "pre-approved" authorities are maintained by companies like Google, Apple, etc. Internally, it has its own "private key"
        - A random string, known as the **server random** string
        - On top of these, the broker now sends a request for the **client's** certificate
    
    3. **Background Check:** Upon receiving the `server hello`, your browser now takes the public certificate, and checks a few things:
        - Check signature against a defined **trust store**, stored somewhere like `/opt/flink/kafka/ssl/truststore.jks`
        - This will check that the broker's certificate was indeed signed by the company's certificate authority inside the file
        - If the broker is not recognise (i.e. the broker's certificate is not valid), Flink kills the connection immediately

    4. **Flink's identity proof:** Flink now reaches into its **key store**, stored somewhere like `/opt/flink/kafka/ssl/keystore.jks` 
        - It sends a copy of its public certificate to the broker
        - Using the key from the key store, it encrypts all handshake data received up to this point using its private key
        - This is sent back to the broker

    5. **Broker validates Flink:** Now using the broker's own trust store, the broker checks the flink app's certificate and signature
        - If all good, the door isu unlocked
    
    6. **Trust Established, Exchange Data:** If the previous step passes, both the broker and flink app know that they are communicating with someone legit. 
        - Flink app generates a third random string, called the **pre-master secret**
        - Use flink's public key (from the certificate) to encrypt this and send to broker
        - Broker decrypts this
    
    7. **Generate session keys:** Now both the flink app and broker have 3 sets of strings; the client random, server random, and pre-master secret. These are used to generate a **master secret**, which is then used to generate **session keys**
        - Now all communication is encrypted with these keys
    